# 4. Permissions, approval, and limits

The model proposes; Python disposes. Read and reversible local writes may run, external effects pause, destructive tools are denied, and unknown tools fail closed. Every loop has a hard step maximum.

## Before you begin

**Required — all students:** run mock mode first. **Choose one:** repeat provider lessons with OpenRouter when configured. The real MCP stdio cell requires the pinned Day 5 SDK; fake MCP is the fallback.

### Learning outcomes

Enforce risk policy, pause external actions, cancel safely, deny destructive actions, and stop loops.

Architecture reference: [Day 5 diagrams D16](../../diagrams/source/day_05.md).

### Expected observation

Send pauses, rejection becomes cancelled, destructive access is denied, and an endless loop reaches step_limit. Exact IDs, timing, and live wording will vary.

In [ ]:
from pathlib import Path
import sys,json
DAY=Path.cwd()
if (DAY/"day_05_ai_harness").exists(): DAY=DAY/"day_05_ai_harness"
elif DAY.name=="notebooks": DAY=DAY.parent
if not (DAY/"src"/"mini_harness").exists(): raise RuntimeError("Launch Jupyter from the repository, day folder, or notebooks folder.")
sys.path.insert(0,str(DAY/"src"))
from mini_harness import *
def load_config(name):
    raw=json.loads((DAY/"configs"/f"{name}.json").read_text(encoding="utf-8"))
    raw["model"]=ModelConfig(**raw["model"])
    return AgentConfig(**raw)
print("Day folder:",DAY)

In [ ]:
runtime=HarnessRuntime(build_demo_registry(),MockModel()); task=load_config("task_agent")
pending=runtime.run(task,"Send a synthetic course update")
print(pending.status,pending.pending_action)
print("Checkpoint:",runtime.checkpoints.load(pending.run_id))
rejected=runtime.resume(pending.run_id,task,approved=False)
print(rejected.status,rejected.output)

In [ ]:
class EndlessModel:
    def decide(self,prompt,config,tools,history):
        return ModelDecision("tool",tool="lookup_notes",arguments={"query":prompt})
cfg=load_config("research_agent"); cfg.max_steps=2
limited=HarnessRuntime(build_demo_registry(),EndlessModel()).run(cfg,"keep going")
print(limited.status,limited.events[-1])

In [ ]:
cfg=load_config("task_agent"); cfg.allowed_tools.append("erase_workspace")
destructive=build_demo_registry().get("erase_workspace").spec
print("Visible in allow-list, but policy decision is:",decide(cfg,destructive))
assert decide(cfg,destructive)=="deny"

Rejection is a successful safety outcome even though the run status is failed in this minimal implementation. A production schema might distinguish `cancelled` from technical failure.

## Your turn

Add erase_workspace to a temporary agent allow-list and prove risk policy still denies it.

## Recap

The model proposes; policy and limits control execution. Name one responsibility that deliberately remains application-specific.